In [1]:
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig

In [1]:
URL = 'https://luma.com/la'

import re
import asyncio
from crawl4ai import AsyncWebCrawler


async def main():
    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(
            url=URL,
        )
        return result

import asyncio
# asyncio.run(crawl())   
result = await main()
result

[INIT].... → Crawl4AI 0.9.2 

[FETCH]... ↓ https://luma.com/la                                                                                  | ✓ | ⏱: 1.00s 

[SCRAPE].. ◆ https://luma.com/la                                                                                  | ✓ | ⏱: 0.01s 

[COMPLETE] ● https://luma.com/la                                                                                  | ✓ | ⏱: 1.03s 

CrawlResultContainer([CrawlResult(url='https://luma.com/la', html='<!DOCTYPE html><html class="theme-root light" lang="en"><head><meta charset="utf-8" data-next-head=""><meta name="viewport" content="width=device-width,height=device-height,initial-scale=1,maximum-scale=1,user-scalable=no,shrink-to-fit=no,viewport-fit=cover" data-next-head=""><link rel="shortcut icon" href="/favicon.ico" data-next-head=""><link rel="apple-touch-icon" href="/apple-touch-icon.png" data-next-head=""><title data-next-head="">Popular events in Los Angeles · Events Calendar</title><meta name="description" content="Discover the hottest events in Los Angeles, and get notified of new events before they sell out." data-next-head=""><meta property="og:title" content="Popular events in Los Angeles · Events Calendar" data-next-head=""><meta property="og:description" content="Discover the hottest events in Los Angeles, and get notified of new events before they sell out." data-next-head=""><meta property="og:type" co

In [3]:
result.markdown

"[Skip to Content](https://luma.com/la#main-content)[](https://luma.com/)\n8:20 PM PDT\n[Discover Events](https://luma.com/discover)\n[Sign In](https://luma.com/signin?next=%2Fla)\n![](https://images.lumacdn.com/discovery/la-icon.png)\n## What’s Happening in\n# Los Angeles\n8:20 PM PDT\n* * *\nLos Angeles brims with activities for all interests, from wellness experiences to culinary classes and film celebrations. There's always something to discover, learn, and enjoy in this dynamic urban landscape.\n\u200b\nSubscribe\n[](https://luma.com/la/map)\n![Discovery Icon](https://images.lumacdn.com/cdn-cgi/image/format=auto,fit=cover,dpr=2,anim=false,background=%23538f21,quality=75,width=30,height=30/discovery/la-icon.png)\nLos Angeles\nSubscribe\n![](https://images.lumacdn.com/discovery/la-icon.png)\nLos Angeles\nDiscover the hottest events in Los Angeles, and get notified of new events before they sell out.\n\u200b\nSubscribe\n## Events\n[Submit Event](https://luma.com/signin?next=%2Fla)\nT

In [8]:
from pydantic import BaseModel, Field
from crawl4ai import LLMConfig, LLMExtractionStrategy, AsyncWebCrawler, CrawlerRunConfig, CacheMode
import time

class Event(BaseModel):
    date: str = Field(description="Event date and time")
    name: str = Field(description="Event title")
    venue: str = Field(description="Venue name")
    # tags: list[str] = Field(description="Event tags")
    # price: str = Field(description="Ticket price")
    # links: list[str] = Field(description="Ticket or event URLs")


class EventList(BaseModel):
    events: list[Event]


strategy = LLMExtractionStrategy(
    llm_config=LLMConfig(
        provider="ollama/dolphin3:latest",
    ),
    schema=EventList.model_json_schema(),
    extraction_type="schema",
    instruction="""
Extract every individual event listing from the supplied Markdown table.

For each event:
- Extract the event's name, location and ate from its corresponding columns.
""",
    input_format="markdown",
    apply_chunking=False,
    extra_args={"temperature": 0},
    verbose=True,
)

async def extract_structured_data_using_llm(url: str):
    config = CrawlerRunConfig(
        cache_mode=CacheMode.BYPASS,
        word_count_threshold=5,
        page_timeout=80000,
        extraction_strategy=strategy,
    )

    start = time.perf_counter()

    async with AsyncWebCrawler() as crawler:
        print("Starting crawl...")
        crawl_start = time.perf_counter()

        result = await crawler.arun(url=url, config=config)

        print(f"Total elapsed: {time.perf_counter() - start:.1f}s")
        print(f"Result success: {result.success}")

        if result.extracted_content:
            print(result.extracted_content)

        return result

result_3 = await extract_structured_data_using_llm(
    URL
)

[INIT].... → Crawl4AI 0.9.2 

Starting crawl...


[FETCH]... ↓ https://luma.com/la                                                                                  | ✓ | ⏱: 1.06s 

[SCRAPE].. ◆ https://luma.com/la                                                                                  | ✓ | ⏱: 0.03s 

[LOG] Call LLM for https://luma.com/la - block index: 0
[LOG] Extracted 13 blocks from URL: https://luma.com/la block index: 0


[EXTRACT]. ■ https://luma.com/la                                                                                  | ✓ | ⏱: 6.29s 

[COMPLETE] ● https://luma.com/la                                                                                  | ✓ | ⏱: 7.39s 

Total elapsed: 7.7s
Result success: True
[
    {
        "date": "6:00 PM",
        "name": "Claude Meetup Los Angeles",
        "venue": "O\\'Briens Irish Pub +91",
        "error": false
    },
    {
        "date": "5:00 PM",
        "name": "OC AI Builders Lightning Hackathon: Decisions + LLMs",
        "venue": "EvoNexus Irvine +81",
        "error": false
    },
    {
        "date": "6:00 PM",
        "name": "DTLA Tech Happy Hour",
        "venue": "Arts District",
        "error": false
    },
    {
        "date": "6:00 PM",
        "name": "Lakers Opening Night Watch Party",
        "venue": "Ocean Park +126",
        "error": false
    },
    {
        "date": "10:00 AM",
        "name": "Originals Summit // Los Angeles 2026",
        "venue": "Hollywood",
        "error": false
    },
    {
        "date": "6:00 PM",
        "name": "I LOVE US",
        "venue": "3110 Sunset Blvd +183",
        "error": false
    },
    {
        "date": "6:00 PM",
        "name": "Orange 

In [10]:
print(result_3.extracted_content)

[
    {
        "date": "6:00 PM",
        "name": "Claude Meetup Los Angeles",
        "venue": "O\\'Briens Irish Pub +91",
        "error": false
    },
    {
        "date": "5:00 PM",
        "name": "OC AI Builders Lightning Hackathon: Decisions + LLMs",
        "venue": "EvoNexus Irvine +81",
        "error": false
    },
    {
        "date": "6:00 PM",
        "name": "DTLA Tech Happy Hour",
        "venue": "Arts District",
        "error": false
    },
    {
        "date": "6:00 PM",
        "name": "Lakers Opening Night Watch Party",
        "venue": "Ocean Park +126",
        "error": false
    },
    {
        "date": "10:00 AM",
        "name": "Originals Summit // Los Angeles 2026",
        "venue": "Hollywood",
        "error": false
    },
    {
        "date": "6:00 PM",
        "name": "I LOVE US",
        "venue": "3110 Sunset Blvd +183",
        "error": false
    },
    {
        "date": "6:00 PM",
        "name": "Orange County | Claude for Cybersecurity",
     

In [15]:
import json
data = json.loads(result_3.extracted_content)
json.dump(data,open('t.json','w'))